# Étape 3 : Valeurs Manquantes

Analyse des mécanismes et imputation sans data leakage.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

from src.feature_engineering import ImputationConfig, add_sdb_indicator

train = pd.read_csv('../data/processed/train_clean.csv', parse_dates=['date_publication'])
test  = pd.read_csv('../data/processed/test_clean.csv',  parse_dates=['date_publication'])
print(f"Train : {train.shape}, Test : {test.shape}")

Train : (1153, 12), Test : (289, 11)


## Analyse des mécanismes de manque

In [2]:
cols_check = ['nb_chambres', 'nb_sdb', 'caracteristiques', 'nb_salons']
missing_pct = {}
for col in cols_check:
    pct_train = train[col].isna().mean() * 100
    pct_test  = test[col].isna().mean()  * 100
    missing_pct[col] = {'train': pct_train, 'test': pct_test}
    print(f"{col:20s} | Train: {pct_train:.1f}% | Test: {pct_test:.1f}%")

nb_chambres          | Train: 1.2% | Test: 2.8%
nb_sdb               | Train: 72.2% | Test: 74.7%
caracteristiques     | Train: 13.6% | Test: 13.5%
nb_salons            | Train: 0.0% | Test: 0.7%


In [ ]:
# Visualisation bar chart horizontal
fig, ax = plt.subplots(figsize=(10, 5))

cols_all = train.columns.tolist()
pcts = [(col, train[col].isna().mean() * 100) for col in cols_all]
pcts = [(c, p) for c, p in pcts if p > 0]
pcts.sort(key=lambda x: x[1], reverse=True)

if pcts:
    labels, values = zip(*pcts)
    bars = ax.barh(labels, values, color='steelblue', edgecolor='white')
    ax.set_xlabel('% valeurs manquantes')
    ax.set_title('Valeurs manquantes par colonne (Train)', fontsize=13)
    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', fontsize=10)
    ax.set_xlim(0, max(values) * 1.15)

plt.tight_layout()
plt.savefig('../outputs/figures/etape3_valeurs_manquantes.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure sauvegardée.")

Figure sauvegardée.


/tmp/ipykernel_363221/1631881197.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Stratégie d'imputation

- `nb_chambres` (MCAR ~1.2%) → médiane par quartier
- `nb_sdb` (MNAR ~72%) → indicatrice `has_sdb_info` + médiane globale train
- `nb_salons` test → médiane par quartier depuis train
- `caracteristiques` → chaîne vide ""

In [4]:
# Créer l'indicatrice AVANT imputation
train = add_sdb_indicator(train)
test  = add_sdb_indicator(test)
print(f"has_sdb_info train: {train['has_sdb_info'].value_counts().to_dict()}")
print(f"has_sdb_info test : {test['has_sdb_info'].value_counts().to_dict()}")

has_sdb_info train: {0: 833, 1: 320}
has_sdb_info test : {0: 216, 1: 73}


In [5]:
# Fit sur le train uniquement
imputer = ImputationConfig()
imputer.fit(train)

print("Médiane nb_chambres par quartier:")
for q, v in imputer.chambres_by_quartier.items():
    print(f"  {q}: {v}")
print(f"\nMédiane nb_sdb (global train): {imputer.sdb_median}")

Médiane nb_chambres par quartier:
  Arafat: 3.0
  Dar Naim: 4.0
  Ksar: 4.0
  Riyadh: 4.0
  Sebkha: 6.5
  Tevragh Zeina: 6.0
  Teyarett: 4.0
  Toujounine: 3.0

Médiane nb_sdb (global train): 2.0


In [6]:
# Appliquer l'imputation
train_imp = imputer.apply_basic(train.copy())
test_imp  = imputer.apply_basic(test.copy())

print("Valeurs manquantes après imputation (TRAIN):")
for col in ['nb_chambres', 'nb_salons', 'nb_sdb']:
    print(f"  {col}: {train_imp[col].isna().sum()} NaN")
    
print("\nValeurs manquantes après imputation (TEST):")
for col in ['nb_chambres', 'nb_salons', 'nb_sdb']:
    print(f"  {col}: {test_imp[col].isna().sum()} NaN")

Valeurs manquantes après imputation (TRAIN):
  nb_chambres: 0 NaN
  nb_salons: 0 NaN
  nb_sdb: 0 NaN

Valeurs manquantes après imputation (TEST):
  nb_chambres: 0 NaN
  nb_salons: 0 NaN
  nb_sdb: 0 NaN


In [7]:
train_imp.to_csv('../data/processed/train_imputed.csv', index=False)
test_imp.to_csv('../data/processed/test_imputed.csv',   index=False)
print("Données imputées sauvegardées.")

Données imputées sauvegardées.
